# 67P/Churyumov-Gerasimenko: fitting the outgassing time delay DT

Reproduces the 67p explore-mode scenario from
<https://empyrean-dynamics.com/explore/67p>.

*This notebook is the twin of [`main.py`](./main.py) and [`main.rs`](./main.rs) — the same scenario, cell by cell.*

What it does:

1. Queries JPL SBDB for 67P. The SBDB solution ships the standard
   Marsden water-sublimation A1/A2/A3 coefficients, the water-ice
   g(r) exponents, and an outgassing time delay DT — the number of
   days by which the peak of the sublimation-driven acceleration
   lags perihelion (thermal inertia carries the activity past the
   sub-solar maximum).
2. Attaches a DT prior to the SBDB orbit, converts it to the
   Cartesian state `refine` solves in (carrying the SBDB covariance
   through the element->Cartesian Jacobian as the Bayesian prior),
   and refines it against 67P's optical astrometry with the wide
   `SolveFor(marsden, dt)` solve, so DT is fit as a free parameter
   jointly with the state and the Marsden coefficients, with an
   honest 1-sigma from the solved covariance. This is the v0.9.0
   fitting surface: DT is no longer a fixed input, it is a
   *measured* quantity.
3. Reports the fitted DT +/- sigma against the SBDB reference.

Authoritative cross-checks (printed inline):

- Orbital period: 6.44 years
- SBDB DT reference: ~+46 d (outgassing lag past perihelion)
- Marsden A1: ~1e-9 AU/d^2

Verified run (empyrean 0.9.0, 11,205 optical obs): fitted
DT = 34.76 +/- 0.43 d, chi2/dof 0.978. That is ~11 d below JPL's
published 45.69 d — far more than the formal sigma, and honestly so:
the formal sigma conditions on the Marsden A-block held at its SBDB
prior, while JPL's DT comes from a joint solve of all four
non-gravitational parameters over a different (radar-inclusive) data
selection. DT is strongly correlated with A1/A2, so the two point
estimates are answers to differently-conditioned questions; the
scenario's point is the *measured uncertainty machinery*, and the
sigma it reports is the honest width of THIS fit, not a claim that
JPL's value is wrong.

DT fitting is a refine-path solve — it needs a DT-primed orbit
(`dt_variance` set, plus a Marsden A1/A2/A3 `covariance` prior for
the joint solve), NOT a cold determine-from-scratch, which the engine
refuses loudly.

## Setup

Install the `empyrean` wheel. On Colab, the first `empyrean.initialize()`
below additionally triggers a one-time ~740 MB kernel download.

In [1]:
%pip install -q empyrean==0.9.0

Note: you may need to restart the kernel to use updated packages.


In [2]:
import empyrean
from empyrean import (
    CartesianCoordinates,
    CartesianOrbits,
    CometaryOrbits,
    NonGravParams,
    ODConfig,
    SolveFor,
    transform_coordinates,
)
import numpy as np


# SBDB water-ice g(r) shape for 67P and the reference outgassing delay.
G_ALPHA, G_R0, G_M, G_N, G_K = 0.1113, 2.808, 2.15, 5.093, 4.6142
SBDB_DT_DAYS = 45.68888251286532
# Wide DT prior (variance, days^2), ~ (20 d)^2: loose enough that the
# astrometry, not the prior, drives the fitted DT.
DT_PRIOR_VAR = 400.0
# Wide Marsden prior: 1-sigma of 1e-9 AU/d^2 on each of A1/A2/A3 --
# the scale of A1 itself, so the astrometry drives the coefficients too.
A_PRIOR_SIGMA = 1.0e-9

empyrean.initialize()

## 1. SBDB query: Marsden A1/A2/A3 + g(r) + reference DT

In [3]:
orbits = empyrean.query_sbdb(["67P"])
ng = orbits.non_grav
print("non-grav coefficients (SBDB):")
print(f"  A1 = {ng.a1.to_numpy()[0]:.3e} AU/d^2")
print(f"  A2 = {ng.a2.to_numpy()[0]:.3e} AU/d^2")
print(f"  A3 = {ng.a3.to_numpy()[0]:.3e} AU/d^2")
print(f"  DT = {SBDB_DT_DAYS:.2f} d  (SBDB reference outgassing lag)")

non-grav coefficients (SBDB):
  A1 = 1.042e-09 AU/d^2
  A2 = -6.740e-11 AU/d^2
  A3 = 2.961e-10 AU/d^2
  DT = 45.69 d  (SBDB reference outgassing lag)


## 2. Prime the orbit with a DT prior

DT fitting is a refine-path solve: the input orbit must carry
`dt_variance` — both the trigger that opens the DT column and the
Bayesian prior. SBDB ships the DT *value*; we attach a wide variance
so the fit is data-driven.

SBDB returns 67P in cometary elements, but `refine` works in the
Cartesian state the differential correction solves for, and it needs
the covariance as its Bayesian prior. `transform_coordinates` carries
the SBDB 6x6 covariance through the element->Cartesian Jacobian; the
non-grav priors (Marsden + DT) are frame/representation invariant, so
we re-attach them unchanged to the Cartesian orbit — nothing is
dropped in the conversion.

In [4]:
non_grav = NonGravParams.from_kwargs(
    a1=ng.a1.to_pylist(),
    a2=ng.a2.to_pylist(),
    a3=ng.a3.to_pylist(),
    model=["marsden"],
    alpha=[G_ALPHA],
    r0=[G_R0],
    m=[G_M],
    n=[G_N],
    k=[G_K],
    dt=[SBDB_DT_DAYS],
    dt_variance=[DT_PRIOR_VAR],  # v0.9.0: opens + priors the DT column
    # v0.9.0: the Marsden wide solve likewise requires a declared
    # A1/A2/A3 prior -- the engine opens the Marsden columns only
    # when the covariance is on the orbit. SBDB ships none for 67P,
    # so we attach a wide diagonal 3x3 (row-major flattened).
    covariance=[np.diag([A_PRIOR_SIGMA**2] * 3).flatten().tolist()],
)

cartesian_coords = transform_coordinates(orbits.coordinates, CartesianCoordinates)
primed = CartesianOrbits.from_kwargs(
    orbit_id=orbits.orbit_id.to_pylist(),
    object_id=orbits.object_id.to_pylist(),
    coordinates=cartesian_coords,
    non_grav=non_grav,
)

## 3. 67P optical astrometry + wide DT refine

In [5]:
obs = empyrean.query_observations(["67P"])
print(f"\n{len(obs)} optical observations")

cfg = ODConfig(solve_for_flags=SolveFor(marsden=True, dt=True))
result = empyrean.refine(primed, obs, config=cfg)

s = result.summary
print(f"Converged:  {result.converged}")
print(f"chi2/dof:   {s.reduced_chi2:.3f}")
print(f'RMS:        RA.cos(d) {s.rms_ra_arcsec:.3f}"  Dec {s.rms_dec_arcsec:.3f}"')


11205 optical observations


Converged:  True
chi2/dof:   0.978
RMS:        RA.cos(d) 1.169"  Dec 1.112"


## 4. Fitted DT +/- 1 sigma (the v0.9.0 measurement)

In [6]:
fitted_dt = result.orbit.non_grav.dt.to_numpy(zero_copy_only=False)[0]
sc = result.solved_covariance
if sc is None or sc.dt_slot is None:
    # Loud failure, not a silent zero: the DT column did not open.
    print("\nDT was not recovered (no DT slot in the solved covariance).")
else:
    dt_sigma = float(np.sqrt(sc.matrix[sc.dt_slot, sc.dt_slot]))
    print(f"\nFitted DT = {fitted_dt:.2f} +/- {dt_sigma:.2f} d")
    print(f"SBDB DT   = {SBDB_DT_DAYS:.2f} d")
    print(
        "(The fit measures the outgassing lag directly from the "
        "astrometry, with an honest sigma from the solved covariance.)"
    )


Fitted DT = 34.76 +/- 0.43 d
SBDB DT   = 45.69 d
(The fit measures the outgassing lag directly from the astrometry, with an honest sigma from the solved covariance.)
